In [3]:
import fitz
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from qdrant_client import QdrantClient
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_88125/3500659790.py:5: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [2]:
load_dotenv()

True

## Load PDF

In [4]:
document_loader = PyMuPDFLoader("Test.pdf")

In [5]:
document = document_loader.load()

## Chunk Data

In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
splitter = SemanticChunker(embeddings)

In [8]:
all_splits = splitter.split_documents(document)

## Embedded Data

In [35]:
from qdrant_client.models import PointStruct, Distance, VectorParams, SparseVectorParams
 


In [10]:
client = QdrantClient(url="http://localhost:6333") 

In [38]:
client.set_sparse_model("Qdrant/bm25")

vector_size = len(embeddings.embed_query("dimension check"))

if client.collection_exists("official_collection"):
    client.delete_collection("official_collection")

collection = client.create_collection(
    collection_name="official_collection",
    vectors_config=VectorParams(
        size=vector_size,
        distance=Distance.COSINE,
    ),
    sparse_vectors_config=client.get_fastembed_sparse_vector_params(),
)

In [22]:
client.get_fastembed_sparse_vector_params()

{'fast-sparse-bm25': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=None, datatype=None), modifier=<Modifier.IDF: 'idf'>)}

In [39]:
dense_points = []
for idx, item in enumerate(all_splits):
    dense_vector = embeddings.embed_query(item.page_content)
    dense_points.append(
        PointStruct(
            id=idx + 1,
            vector=dense_vector,
            payload={
                "text": item.page_content,
                "metadata": item.metadata,
            },
        )
    )

client.upsert(
    collection_name="official_collection",
    points=dense_points,
    wait=True,
    )

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [43]:
question = "What is the main topic of the document?"
query_vector = embeddings.embed_query(question)

hits = client.query_points(
    collection_name="official_collection",
    query=query_vector,
    limit=3,
    with_payload=True,
 )

for point in hits.points:
    print(f"ID: {point.id}, score: {point.score:.4f}")
    print(point.payload.get("text", "")[:220])
    print("-" * 80)

ID: 68, score: 0.2657
57, No. 3, Article 55. Publication date: November 2024.
--------------------------------------------------------------------------------
ID: 57, score: 0.2657
57, No. 3, Article 55. Publication date: November 2024.
--------------------------------------------------------------------------------
ID: 9, score: 0.2657
57, No. 3, Article 55. Publication date: November 2024.
--------------------------------------------------------------------------------


In [34]:
embeddings.embed_query(all_splits[0].page_content)

[-0.026275634765625,
 0.046661376953125,
 -0.0111541748046875,
 -0.005950927734375,
 0.05474853515625,
 0.0220947265625,
 -0.004421234130859375,
 0.01654052734375,
 -0.01084136962890625,
 0.07208251953125,
 0.04248046875,
 -0.0692138671875,
 -0.0009565353393554688,
 -0.004856109619140625,
 -0.035064697265625,
 0.0292205810546875,
 -0.003032684326171875,
 -0.0462646484375,
 -0.0002727508544921875,
 -0.04266357421875,
 -0.009033203125,
 -0.005542755126953125,
 -0.0085906982421875,
 0.03118896484375,
 -0.0169219970703125,
 0.0062255859375,
 0.00835418701171875,
 -0.0167999267578125,
 -0.018798828125,
 0.033599853515625,
 0.043060302734375,
 0.0002353191375732422,
 -0.01253509521484375,
 -0.0222015380859375,
 0.00064849853515625,
 -0.01107025146484375,
 -0.005184173583984375,
 0.03790283203125,
 -0.00518035888671875,
 0.0232696533203125,
 0.03240966796875,
 -0.058563232421875,
 0.01287078857421875,
 -0.0007977485656738281,
 0.01407623291015625,
 -0.01047515869140625,
 -0.02130126953125,
 -